# MEG Stroke Intervention - Model Analysis

This notebook analyzes the trained MEGStrokeNet model:

- Architecture summary and parameter count
- Layer-by-layer weight distributions
- Inference performance on test data
- Quantization error analysis
- Arduino deployment feasibility

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from models.meg_stroke_net import MEGStrokeNet

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load trained model
from pathlib import Path

model = MEGStrokeNet()
model_path = Path('..') / 'models' / 'trained_model.pth'

if model_path.exists():
    state = torch.load(str(model_path), map_location='cpu', weights_only=True)
    if 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
        print(f'Loaded model (val_loss={state.get("best_val_loss", "N/A")})')
    else:
        model.load_state_dict(state)
else:
    print('No trained model found - using random weights')

model.eval()
print(f'Total parameters: {model.count_parameters():,}')
print(f'Output names: {MEGStrokeNet.OUTPUT_NAMES}')

In [ ]:
# Architecture summary
print('Layer-by-layer parameter count:')
print(f'{"Layer":<35} {"Shape":<25} {"Params":>8}')
print('-' * 70)
for name, param in model.named_parameters():
    print(f'{name:<35} {str(list(param.shape)):<25} {param.numel():>8,}')

# Forward pass test
x = MEGStrokeNet.get_example_input(batch_size=1)
with torch.no_grad():
    y = model(x)
print(f'\nInput:  {x.shape}')
print(f'Output: {y.shape} = {y.squeeze().tolist()}')

In [ ]:
# Weight distributions per layer
params = [(name, p.detach().numpy().ravel()) for name, p in model.named_parameters()]

fig, axes = plt.subplots(2, (len(params) + 1) // 2, figsize=(16, 8))
axes = axes.ravel()

for idx, (name, weights) in enumerate(params):
    if idx >= len(axes):
        break
    ax = axes[idx]
    ax.hist(weights, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(name, fontsize=9)
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Value', fontsize=8)

# Hide unused axes
for idx in range(len(params), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Weight Distributions by Layer', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Quantization error analysis
quant_errors = []
for name, param in model.named_parameters():
    w = param.detach().numpy().astype(np.float32)
    abs_max = np.abs(w).max()
    scale = 127.0 / (abs_max + 1e-12)
    w_int8 = np.clip(np.round(w * scale), -128, 127).astype(np.int8)
    w_recovered = w_int8.astype(np.float32) / scale
    error = np.abs(w - w_recovered)
    quant_errors.append((name, error.ravel()))

fig, ax = plt.subplots(figsize=(12, 5))
bp_data = [e for _, e in quant_errors]
bp_labels = [n.replace('.', '\n') for n, _ in quant_errors]
ax.boxplot(bp_data, labels=bp_labels)
ax.set_ylabel('Absolute Quantization Error')
ax.set_title('INT8 Quantization Error by Layer')
plt.xticks(fontsize=7, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Arduino deployment feasibility
timing = model.inference_time_estimate()

param_count = model.count_parameters()
int8_bytes = param_count
fp32_bytes = param_count * 4

print('=== Arduino Deployment Feasibility ===')
print(f'Parameters:       {param_count:,}')
print(f'FP32 size:        {fp32_bytes:,} bytes ({fp32_bytes/1024:.1f} KB)')
print(f'INT8 size:         {int8_bytes:,} bytes ({int8_bytes/1024:.1f} KB)')
print(f'Flash budget:     32 KB -> {"OK" if int8_bytes + 4096 < 32768 else "OVER"}')
print(f'\nEstimated FLOPs:  {timing["estimated_flops"]:,}')
print(f'Est. latency:     {timing["estimated_latency_ms"]} ms (at 16 MHz)')
print(f'Target:           < 50 ms -> {"OK" if timing["estimated_latency_ms"] < 50 else "OVER"}')

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Memory comparison
ax1.bar(['FP32', 'INT8', 'Budget'], [fp32_bytes/1024, int8_bytes/1024, 32],
        color=['#F44336', '#4CAF50', '#9E9E9E'])
ax1.set_ylabel('KB')
ax1.set_title('Flash Memory Usage')
ax1.axhline(y=32, color='black', linestyle='--', alpha=0.5)

# Latency
ax2.bar(['Estimated', 'Target'], [timing['estimated_latency_ms'], 50],
        color=['#4CAF50' if timing['estimated_latency_ms'] < 50 else '#F44336', '#9E9E9E'])
ax2.set_ylabel('ms')
ax2.set_title('Inference Latency (16 MHz)')

plt.tight_layout()
plt.show()